# Preparation

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

NEO4J_URI = "bolt://localhost:17687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "X" 
OLLAMA_MODEL = "qwen3-embedding:4b"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [ ]:
index_name = "documents"

with driver.session() as session:
    session.run(f"DROP INDEX {index_name} IF EXISTS")
    
    session.run(f"""
        CREATE VECTOR INDEX {index_name} IF NOT EXISTS
        FOR (p:product) ON (p.embedding)
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: 2560,
            `vector.similarity_function`: 'cosine'
          }}
        }}
    """)

In [ ]:
ollama_embedder = OllamaEmbeddings(model=OLLAMA_MODEL)

retriever = VectorRetriever(
    driver=driver,
    index_name=index_name,
    embedder=ollama_embedder,
    return_properties=["name", "document"]
)

# Suchanfrage definieren
query = "I would like a toy for my cat!"

# Suche durchführen (Top k ähnlichste Ergebnisse)
search_results = retriever.search(query_text=query, top_k=5)

print(f"Suchanfrage: '{query}'\n")
print("Ergebnisse:")
for i, result in enumerate(search_results.items, 1):
    print(i, result)
    print(f"[{i}] Score: {result.metadata.get('score'):.4f}")